# neo4j-nano demo

DataFrames in, Cypher out. Neo4j Virtual Graphs embedded directly in Python — no Docker, no server, no network.

In [ ]:
%pip install pandas jpype1 neo4j-nano

  Using cached pandas-2.3.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached pandas-2.3.3-cp310-cp310-macosx_11_0_arm64.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 1.7 MB/s eta 0:00:00-:--:--
Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
from neo4j_nano import GraphEngine

## Load data from CSV

In [6]:
movies_df = pd.read_csv("data/movies.csv")
people_df = pd.read_csv("data/people.csv")
acted_in_df = pd.read_csv("data/acted_in.csv")
directed_df = pd.read_csv("data/directed.csv")

## Build graph and start engine

In [7]:
engine = GraphEngine(accept_license=True)

# Nodes
engine.add_nodes(movies_df, label="Movie", id_column="movie_id")
engine.add_nodes(people_df, label="Person", id_column="person_id")

# Relationships
engine.add_relationships(acted_in_df, type="ACTED_IN",
    source_column="person_id", source_label="Person",
    target_column="movie_id", target_label="Movie")
engine.add_relationships(directed_df, type="DIRECTED",
    source_column="person_id", source_label="Person",
    target_column="movie_id", target_label="Movie")

engine.start()

Starting JVM with Neo4j + SQLite JDBC classpath...
Neo4j embedded started with Virtual Graphs.


## Query: Movies after 2000

In [9]:
pd.DataFrame(engine.query("""
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    WHERE m.release_year > 2000
    RETURN p.name AS actor, m.title AS movie, m.release_year AS year
"""))

,actor,movie,year
0,Keanu Reeves,John Wick,2014


## Query: All paths

In [10]:
pd.DataFrame(engine.query("""
    MATCH (p:Person)-[r]->(m:Movie)
    RETURN p.name AS person, type(r) AS relationship, m.title AS movie
    ORDER BY person, movie
"""))

,person,relationship,movie
0,Carrie-Anne Moss,ACTED_IN,The Matrix
1,Keanu Reeves,ACTED_IN,John Wick
2,Keanu Reeves,ACTED_IN,Speed
3,Keanu Reeves,ACTED_IN,The Matrix
4,Lana Wachowski,DIRECTED,The Matrix


## Query: Co-actors

In [11]:
pd.DataFrame(engine.query("""
    MATCH (p1:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(p2:Person)
    WHERE p1.name < p2.name
    RETURN p1.name AS actor1, p2.name AS actor2, m.title AS movie
"""))

,actor1,actor2,movie
0,Carrie-Anne Moss,Keanu Reeves,The Matrix


## Cleanup

In [12]:
engine.stop()